In [5]:
import duckdb
from pathlib import Path

DATA_GLOB = str(Path("../data/raw/*.csv"))
con = duckdb.connect()

summary = con.sql(f"""
    SELECT
        COUNT(*)                          AS total_rows,
        COUNT(DISTINCT serial_number)     AS drives,
        MIN(date)                         AS first_day,
        MAX(date)                         AS last_day,
        SUM(CAST(failure AS INTEGER))     AS failure_rows
    FROM read_csv('{DATA_GLOB}', all_varchar=true, union_by_name=true)
""").df()
summary

,total_rows,drives,first_day,last_day,failure_rows
0,31968003,359101,2026-04-01,2026-06-30,1522.0


In [6]:
con.sql(f"""
    COPY (
        SELECT
            CAST(date AS DATE)                 AS date,
            serial_number,
            model,
            TRY_CAST(capacity_bytes AS BIGINT) AS capacity_bytes,
            CAST(failure AS INTEGER)           AS failure
        FROM read_csv('{DATA_GLOB}', all_varchar=true, union_by_name=true)
    ) TO '../data/drive_index.parquet' (FORMAT PARQUET)
""")

models = con.sql("""
    SELECT model,
           COUNT(DISTINCT serial_number)  AS drives,
           SUM(failure)                   AS failures,
           ROUND(100.0 * SUM(failure) / COUNT(DISTINCT serial_number), 2) AS pct_failed
    FROM '../data/drive_index.parquet'
    GROUP BY model
    HAVING SUM(failure) > 0
    ORDER BY failures DESC
    LIMIT 15
""").df()
models

,model,drives,failures,pct_failed
0,HGST HUH721212ALN604,9694,180.0,1.86
1,TOSHIBA MG07ACA14TA,37171,179.0,0.48
2,ST12000NM0008,18521,145.0,0.78
3,ST8000NM0055,13112,112.0,0.85
4,HGST HUH721212ALE604,13196,110.0,0.83
5,TOSHIBA MG08ACA16TA,39932,102.0,0.26
6,ST24000NM002H,9631,82.0,0.85
7,WDC WUH721816ALE6L4,26286,74.0,0.28
8,TOSHIBA MG10ACA20TE,25270,64.0,0.25
9,WDC WUH722222ALE6L4,45665,63.0,0.14
